In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
email=pd.read_csv(r"A:\lenovo\Desktop\datasets\Phishing_Email.csv")
email.head()

,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [4]:
email.drop('Unnamed: 0',axis=1,inplace=True)

In [5]:
email.head()

,Email Text,Email Type
0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,the other side of * galicismos * * galicismo *...,Safe Email
2,re : equistar deal tickets are you still avail...,Safe Email
3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [6]:
email.isnull().sum()

Email Text    16
Email Type     0
dtype: int64

In [7]:
email=email.dropna()

In [8]:
email.isnull().sum()

Email Text    0
Email Type    0
dtype: int64

In [9]:
email['Email Type'].value_counts()

Email Type
Safe Email        11322
Phishing Email     7312
Name: count, dtype: int64

In [10]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    return text


In [11]:
email["clean_text"] = email["Email Text"].astype(str).apply(clean_text)

In [12]:
email["label"] = email['Email Type'].map({
    "Safe Email": 0,
    "Phishing Email": 1
})

print(email["label"].value_counts())


label
0    11322
1     7312
Name: count, dtype: int64


In [13]:
texts = email["clean_text"].tolist()
labels = email["label"].tolist()


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels,
    test_size=0.2,
    stratify=labels,
    random_state=39
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=13
)


In [15]:
X_test

['iatl  israel association for theoretical linguistics forteenth annual conference we enclose details about accommodation  travel and the schedule for iatl   to be held be held in ben gurion university  beer sheva  on june      we hope you will be able to join us  accommodation  people wishing to book accommodation should contact ariel cohen  arikc  bgumail  bgu  ac  il  asap  there are three main options   beersheva hilton  single    double    suite    all prices per night and include breakfast  the prices do not include   vat    neot midbar hotel  single    double    triple    lunch    dinner    at the red sand grill restaurant    bgu dorms  shared apartment   nis  approx    for one night  or  nis     approx  for up to a week  own apartment  suitable for a couple    nis for one night  or  nis for up to a week  travel  beersheva is well connected to both telaviv and jerusalem by fast  air conditioned egged buses  ask driver for ben gurion university stop  once you  re safely off the b

In [16]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_WORDS = 10000
MAX_LEN = 200

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(
    tokenizer.texts_to_sequences(X_train),
    maxlen=MAX_LEN,
    padding="post"
)

X_val_seq = pad_sequences(
    tokenizer.texts_to_sequences(X_val),
    maxlen=MAX_LEN,
    padding="post"
)

X_test_seq = pad_sequences(
    tokenizer.texts_to_sequences(X_test),
    maxlen=MAX_LEN,
    padding="post"
)


In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(MAX_WORDS, 128, input_length=MAX_LEN),
    LSTM(128, return_sequences=False),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()




Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 200, 128)          1280000   
                                                                 
 lstm (LSTM)                 (None, 128)               131584    
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 1411713 (5.39 MB)
Trainable params: 1411713 (5.39 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [18]:
import numpy as np

X_train_seq = np.array(X_train_seq)
X_val_seq   = np.array(X_val_seq)

y_train = np.array(y_train)
y_val   = np.array(y_val)


In [19]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train
)

class_weight_dict = {
    0: class_weights[0],
    1: class_weights[1]
}


In [20]:
history = model.fit(
    X_train_seq,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_val_seq, y_val),
    class_weight=class_weight_dict
)


Epoch 1/5


466/466 [==============================] - 46s 94ms/step - loss: 0.5533 - accuracy: 0.7019 - val_loss: 0.5248 - val_accuracy: 0.7123
Epoch 2/5
466/466 [==============================] - 47s 101ms/step - loss: 0.2807 - accuracy: 0.8989 - val_loss: 0.3255 - val_accuracy: 0.9023
Epoch 3/5
466/466 [==============================] - 46s 98ms/step - loss: 0.3290 - accuracy: 0.8907 - val_loss: 0.4986 - val_accuracy: 0.8170
Epoch 4/5
466/466 [==============================] - 45s 97ms/step - loss: 0.3144 - accuracy: 0.8933 - val_loss: 0.2796 - val_accuracy: 0.9136
Epoch 5/5
466/466 [==============================] - 43s 91ms/step - loss: 0.2234 - accuracy: 0.9272 - val_loss: 0.2266 - val_accuracy: 0.9163


In [21]:
model.save("new_email_lstm_model.keras")


In [22]:
import pickle

with open("new_email_lstm_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)


In [ ]:
#95.81 accuracy